### Importing Libraries

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from sklearn.metrics import classification_report

KeyboardInterrupt: 

### Data Loading

In [2]:
dataset = load_dataset('csv', data_files={
    'train': 'cleaned_train_data.csv', 
    'test': 'cleaned_test_data.csv'
})
print(dataset['train'].column_names)

['Sentence', 'Emotion', 'Label']


### Data preprocessing

In [3]:
# Rename columns to match expected format
dataset = dataset.rename_column("Sentence", "text")
dataset = dataset.class_encode_column("Emotion")  # converts to integers if needed
dataset = dataset.rename_column("Label", "labels")

### Tokenizer Initialization

In [4]:
model_name = 'microsoft/deberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
max_length = 128 

### Tokenization

In [5]:
def preprocess_function(examples):
    return tokenizer(
        examples['text'], 
        truncation=True, 
        padding='max_length', 
        max_length=max_length
    )

tokenized_datasets = dataset.map(preprocess_function, batched=True)

### Model Initialization

In [6]:
num_labels = len(set(dataset['train']['labels']))
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=num_labels
)

Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### Metrics Setup

In [7]:
accuracy = evaluate.load('accuracy')
f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy.compute(predictions=predictions, references=labels)['accuracy'],
        'f1': f1.compute(predictions=predictions, references=labels, average='weighted')['f1'],
    }

### Training Arguments

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./deberta-finetuned',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_dir='./logs',
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    warmup_steps=500,
    max_grad_norm=1.0,
    bf16=False,
    fp16=False, # changed to False to avoid overflow error
    seed=42,
)


### Trainer Initialization

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

C:\Users\mailo\AppData\Local\Temp\ipykernel_36928\2658110253.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


### Model Training

In [10]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.900600,0.883676,0.680165,0.675153
2,0.828300,0.821504,0.698347,0.693921
3,0.524400,0.903109,0.709917,0.704223
4,0.341900,1.020201,0.713223,0.711095
5,0.195500,1.270602,0.705785,0.704308
6,0.157000,1.417813,0.713223,0.712460
7,0.149800,1.705716,0.700000,0.696040
8,0.063300,1.848368,0.712397,0.708847
9,0.107800,1.935376,0.718182,0.715187
10,0.035100,1.949000,0.719008,0.717432


TrainOutput(global_step=14810, training_loss=0.34805502151815093, metrics={'train_runtime': 4069.5278, 'train_samples_per_second': 58.211, 'train_steps_per_second': 3.639, 'total_flos': 1.815817584721152e+16, 'train_loss': 0.34805502151815093, 'epoch': 10.0})

### Model Evaluation & Classification Report

In [11]:
predictions_output = trainer.predict(tokenized_datasets['test'])
preds = np.argmax(predictions_output.predictions, axis=-1)
labels = predictions_output.label_ids

In [12]:
target_names = [str(i) for i in range(num_labels)]

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(labels, preds, target_names=target_names))


CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.73      0.67      0.70       155
           1       0.69      0.65      0.67        72
           2       0.60      0.44      0.51        59
           3       0.75      0.73      0.74       278
           4       0.74      0.79      0.77       391
           5       0.62      0.70      0.66       129
           6       0.73      0.73      0.73       126

    accuracy                           0.72      1210
   macro avg       0.70      0.67      0.68      1210
weighted avg       0.72      0.72      0.72      1210

